## Following kernels will load source name text files into a filtered ARCHS4 database

### Function to load sample names from text file

In [1]:
import glob
import sys
import h5py
import archs4py as a4
import re
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

In [2]:
def load_sample_names(txt_file_path):
    """Load sample names from a text file"""
    with open(txt_file_path, 'r') as f:
        sample_names = [line.strip() for line in f if line.strip()]
    return sample_names

In [3]:
%cd /scratch/by2372/pipeline/ARCHS4_Macrophages/Source_name_filtering/
class_sample_dicts = list()
base_dict = dict()

file_path = "/scratch/by2372/pipeline/ARCHS4_Macrophages/Source_name_filtering/"
Monocyte_file = "Monocyte3.txt"
Macrophage_file = "Macrophage3.txt"

# Extract the Monocyte and Macrophage sample names and put them into base_dict
base_dict["Monocyte"] = load_sample_names(file_path + Monocyte_file)
base_dict["Macrophage"] = load_sample_names(file_path + Macrophage_file)
class_sample_dicts.append(base_dict)

for filename in glob.glob("*3.txt"):

    this_path = file_path + filename
    sample_names = load_sample_names(this_path)
    class_name = filename.replace("3.txt","")

    if class_name == "Monocyte" or class_name == "Macrophage":
        continue
    else:
        this_dict = dict()
        this_dict["Monocyte"] = base_dict["Monocyte"]
        this_dict["Macrophage + " + class_name] = base_dict["Macrophage"] + sample_names
        class_sample_dicts.append(this_dict)

/scratch/by2372/pipeline/ARCHS4_Macrophages/Source_name_filtering


In [4]:
len(class_sample_dicts)

6

In [5]:
for d in class_sample_dicts:
    keys = list(d.keys())
    print(keys[1])

Macrophage
Macrophage + Kupffer
Macrophage + Langerhans
Macrophage + Microglia
Macrophage + Osteoclast
Macrophage + alveolar


### Filters human_tpm_v2.2_NOSC.h5 into a filtered file containing source names in class_sample_dict

In [6]:
transcript_file_path = "/scratch/by2372/pipeline/ARCHS4_Macrophages/human_tpm_v2.2_NOSC.h5"
a4.ls(transcript_file_path)

data                      
│ expression            float32 | (271742, 661840)
meta                      
│ samples                   
│   characteristics_ch1   str    | (661840,)
│   geo_accession         str    | (661840,)
│   source_name_ch1       str    | (661840,)
│ transcripts               
│   ensembl_id            str    | (271742,)
│   symbol                str    | (271742,)


In [7]:
for d in class_sample_dicts:

    mac_key = list(d.keys())[1].replace(" + ","_")
    out_file_name = "filtered_ARCHS4_" + mac_key + ".h5"

    with h5py.File(transcript_file_path, "r") as fin, h5py.File(out_file_name, 'w') as fout:
        
        # copy the ensembl_ids and gene symbols to the new h5 file
        fin.copy("meta/transcripts/ensembl_id", fout)
        fin.copy("meta/transcripts/symbol", fout) 
        
        # Navigate to the metadata group for source names
        source_name_ds = fin['meta/samples/source_name_ch1']
    
        # Read all names and decode bytes to strings
        source_names = source_name_ds[:]
        if isinstance(source_names[0], bytes):
            source_names = [s.decode('utf-8') for s in source_names]
    
        # Get dataset info
        expr_dataset = fin['data/expression']
        n_genes, n_samples = expr_dataset.shape
    
        filtered_dataset = fout.create_dataset(
            "data/filtered_expression",
            (n_genes, 0),                    # start with 0 samples
            maxshape=(n_genes, None),        # allow dynamic extension
            dtype=expr_dataset.dtype,
            chunks=True
        )
        
        class_name_dataset = fout.create_dataset(
            "class_name",
            (0,),                            # start empty
            maxshape=(None,),                # allow appending
            dtype=source_name_ds.dtype,
            chunks=True
        )
    
        current_pos = 0
    
        for key, value in d.items():
    
            print("Matching indices for " + str(key))
    
            
            matched_indices = []
            class_names = []
            
            for filt_name in set(value): 
                matched_indices.extend([
                    i for i, n in enumerate(source_names)
                    if n.lower() == filt_name.lower()
                ])
    
            matched_indices.sort()
            n_new = len(matched_indices)
            print(str(n_new))
            cur_class_names = [key] * n_new
            class_names.append(cur_class_names)
    
            filtered_dataset.resize((n_genes, current_pos + n_new))
            class_name_dataset.resize((current_pos + n_new,))
            
            # Process in chunks of genes instead of loading all at once
            chunk_size = 1000  # Adjust based on your memory
            iteration = 0
            for start_gene in range(0, n_genes, chunk_size):
                end_gene = min(start_gene + chunk_size, n_genes)
                
                # Read chunk of data for matched samples only
                chunk_data = expr_dataset[start_gene:end_gene, matched_indices]
                
                # Write to output
                filtered_dataset[start_gene:end_gene, current_pos:current_pos + n_new] = chunk_data
                iteration = iteration + 1
                print("Iteration " + str(iteration)+ " is done.")
                sys.stdout.flush()
    
    
            class_name_dataset[current_pos:current_pos + n_new] = class_names
            
            current_pos += n_new
        

Matching indices for Monocyte
5497
Iteration 1 is done.
Iteration 2 is done.
Iteration 3 is done.
Iteration 4 is done.
Iteration 5 is done.
Iteration 6 is done.
Iteration 7 is done.
Iteration 8 is done.
Iteration 9 is done.
Iteration 10 is done.
Iteration 11 is done.
Iteration 12 is done.
Iteration 13 is done.
Iteration 14 is done.
Iteration 15 is done.
Iteration 16 is done.
Iteration 17 is done.
Iteration 18 is done.
Iteration 19 is done.
Iteration 20 is done.
Iteration 21 is done.
Iteration 22 is done.
Iteration 23 is done.
Iteration 24 is done.
Iteration 25 is done.
Iteration 26 is done.
Iteration 27 is done.
Iteration 28 is done.
Iteration 29 is done.
Iteration 30 is done.
Iteration 31 is done.
Iteration 32 is done.
Iteration 33 is done.
Iteration 34 is done.
Iteration 35 is done.
Iteration 36 is done.
Iteration 37 is done.
Iteration 38 is done.
Iteration 39 is done.
Iteration 40 is done.
Iteration 41 is done.
Iteration 42 is done.
Iteration 43 is done.
Iteration 44 is done.
Iterat

# Following code with filter the databases for TF genes only

In [8]:
tf_names = load_sample_names("/scratch/by2372/pipeline/tf_names.txt")
for d in class_sample_dicts:

    mac_key = list(d.keys())[1].replace(" + ","_")
    in_file_name = "filtered_ARCHS4_" + mac_key + ".h5"
    out_file_name = "filtered_ARCHS4_TFs_" + mac_key + ".h5"

    # Read in the filtered ARCHS4 h5 and write out to a new h5
    with h5py.File(in_file_name, "r") as fin, h5py.File(out_file_name, 'w') as fout:
    
        # Copy the class names into the new h5
        fin.copy("class_name", fout) 
        # Read in the gene symbol and ensembl IDs from the input h5 file
        symbols = fin["symbol"]
        ensembl_id = fin["ensembl_id"]
    
        # unpack the gene symbol names and decode bytes to strings
        symbols_unpack = symbols[:]
        if isinstance(symbols_unpack[0], bytes):
            symbols_unpack = [s.decode('utf-8') for s in symbols_unpack]
    
        # unapck the ensembl IDs and decode bytes to strings
        ensembl_unpack = ensembl_id[:]
        if isinstance(ensembl_unpack[0], bytes):
            ensembl_unpack = [s.decode('utf-8') for s in ensembl_unpack]
    
        # initialize the total number of genes in the input h5, an empty matched index list and the processing chunk size
        n_genes = len(symbols)
        tf_matched_indices = []
        chunk_size = 1000
    
        # iterate through indices of all genes in the input h5 by chunking through them
        for start_gene in range(0, n_genes, chunk_size):
            end_gene = min(start_gene + chunk_size, n_genes)
            chunk_symbols = symbols_unpack[start_gene:end_gene]
    
            # find the indices that match the list of known human TFs
            for i, sym in enumerate(chunk_symbols):
                if sym in tf_names:
                    idx = start_gene + i
                    tf_matched_indices.append(idx)
    
        tf_matched_indices.sort()
        print("Matched indices for TFs have been found")
        
        # Create output dataset for gene symbol in the output h5
        new_sym_dataset = fout.create_dataset(
            "symbol", 
            (len(tf_matched_indices), ),
            dtype=symbols.dtype,
            chunks=True
        )
    
        # Create output dataset for ensembl ID in the output h5
        new_ensembl_dataset = fout.create_dataset(
            "ensembl_id", 
            (len(tf_matched_indices), ),
            dtype=ensembl_id.dtype,
            chunks=True
        )
    
        # convert symbols and ensembl IDs to np arrays
        symbols_arr = np.array(symbols_unpack)
        ensembl_arr = np.array(ensembl_unpack)
    
        # have only the matched indices be the new symbols and ensembl ID datasets
        new_sym_dataset[...] = symbols_arr[tf_matched_indices]
        new_ensembl_dataset[...] = ensembl_arr[tf_matched_indices]
    
        # Get dataset info
        expr_dataset = fin['data/filtered_expression']
        n_genes, n_samples = expr_dataset.shape
    
        # Create output dataset
        filtered_dataset = fout.create_dataset(
            "data/filtered_expression", 
            (len(tf_matched_indices), n_samples),
            dtype=expr_dataset.dtype,
            chunks=True
        )
    
        # do a chunking approach to chunk through the number of samples in 1000 chunks 
        iteration = 0
        for start_sample in range(0, n_samples, chunk_size):
            end_sample = min(start_sample + chunk_size, n_samples)
            data_chunk = expr_dataset[tf_matched_indices, start_sample:end_sample]
            filtered_dataset[:, start_sample:end_sample] = data_chunk
            print("Iteration " + str(iteration)+ " is done.")
            iteration += 1


Matched indices for TFs have been found
Iteration 0 is done.
Iteration 1 is done.
Iteration 2 is done.
Iteration 3 is done.
Iteration 4 is done.
Iteration 5 is done.
Iteration 6 is done.
Iteration 7 is done.
Iteration 8 is done.
Iteration 9 is done.
Matched indices for TFs have been found
Iteration 0 is done.
Iteration 1 is done.
Iteration 2 is done.
Iteration 3 is done.
Iteration 4 is done.
Iteration 5 is done.
Iteration 6 is done.
Iteration 7 is done.
Iteration 8 is done.
Iteration 9 is done.
Matched indices for TFs have been found
Iteration 0 is done.
Iteration 1 is done.
Iteration 2 is done.
Iteration 3 is done.
Iteration 4 is done.
Iteration 5 is done.
Iteration 6 is done.
Iteration 7 is done.
Iteration 8 is done.
Iteration 9 is done.
Matched indices for TFs have been found
Iteration 0 is done.
Iteration 1 is done.
Iteration 2 is done.
Iteration 3 is done.
Iteration 4 is done.
Iteration 5 is done.
Iteration 6 is done.
Iteration 7 is done.
Iteration 8 is done.
Iteration 9 is done.


In [12]:
for d in class_sample_dicts:

    mac_key = list(d.keys())[1].replace(" + ","_")
    TF_file_name = "filtered_ARCHS4_TFs_" + mac_key + ".h5"
    print(TF_file_name)
    a4.ls("/scratch/by2372/pipeline/ARCHS4_Macrophages/Source_name_filtering/" + TF_file_name)
    print("\n")

filtered_ARCHS4_TFs_Macrophage.h5
class_name            str    | (9680,)
data                      
│ filtered_expression   float32 | (13146, 9680)
ensembl_id            str    | (13146,)
symbol                str    | (13146,)


filtered_ARCHS4_TFs_Macrophage_Kupffer.h5
class_name            str    | (9715,)
data                      
│ filtered_expression   float32 | (13146, 9715)
ensembl_id            str    | (13146,)
symbol                str    | (13146,)


filtered_ARCHS4_TFs_Macrophage_Langerhans.h5
class_name            str    | (9828,)
data                      
│ filtered_expression   float32 | (13146, 9828)
ensembl_id            str    | (13146,)
symbol                str    | (13146,)


filtered_ARCHS4_TFs_Macrophage_Microglia.h5
class_name            str    | (10352,)
data                      
│ filtered_expression   float32 | (13146, 10352)
ensembl_id            str    | (13146,)
symbol                str    | (13146,)


filtered_ARCHS4_TFs_Macrophage_Osteoclast.h5
clas

# The code below will create training and testing datasets

In [14]:
def chunk_samples(chunk_size, n_samples, n_genes, output, input):
    iteration = 0
    for start_sample in range(0, n_samples, chunk_size):
        end_sample = min(start_sample + chunk_size, n_samples)

        for start_gene in range(0, n_genes, chunk_size):
            end_gene = min(start_gene + chunk_size, n_genes)

            # Read a chunk of the input matrix
            data_chunk = input[start_sample:end_sample, start_gene:end_gene]

            # Write the chunk to the output dataset
            output[start_sample:end_sample, start_gene:end_gene] = data_chunk

        print("Iteration " + str(iteration) + " (samples " + str(start_sample) + "-" + str(end_sample) + ") is done.")
        iteration += 1

In [15]:
def chunk_class(chunk_size,n_samples,output,input):
    iteration = 0
    for start_sample in range(0, n_samples, chunk_size):
        end_sample = min(start_sample + chunk_size, n_samples)
        data_chunk = input[start_sample:end_sample]
        output[start_sample:end_sample] = data_chunk
        print("Iteration " + str(iteration)+ " is done.")
        iteration += 1

In [16]:
for d in class_sample_dicts:

    mac_key = list(d.keys())[1].replace(" + ","_")
    in_file_name = "filtered_ARCHS4_TFs_" + mac_key + ".h5"
    out_train_name = "filtered_ARCHS4_train_" + mac_key + ".h5"
    out_test_name = "filtered_ARCHS4_test_" + mac_key + ".h5"

    # Make Rows be the samples and the columns be the transcripts
    # make it into a data frame and export as H5
    with h5py.File(in_file_name, "r") as fin, h5py.File(out_train_name, 'w') as ftrain, h5py.File(out_test_name, 'w') as ftest:
    
        # Navigate to the metadata group for class names, gene symbols, and ensembl ID
        class_name_ds = fin['class_name']
        ensembl_id = fin["ensembl_id"]
        symbols = fin["symbol"]
    
        # Read all names and decode bytes to strings
        symbols_unpack = symbols[:]
        if isinstance(symbols_unpack[0], bytes):
            symbols_unpack = [s.decode('utf-8') for s in symbols_unpack]
    
        # Read all names and decode bytes to strings
        class_names = class_name_ds[:]
        if isinstance(class_names[0], bytes):
            class_names = [s.decode('utf-8') for s in class_names]
    
        ensembl_unpack = ensembl_id[:]
        if isinstance(ensembl_unpack[0], bytes):
            ensembl_unpack = [s.decode('utf-8') for s in ensembl_unpack]
    
        # Navigate to the metadata group for the expression data
        expression = fin["data/filtered_expression"]
    
        # Transpose expression data so that rows are samples and columns are transcripts
        expression_data = expression[:, :] 
        expression_data = expression_data.T
        print("Expression data has been transposed\n")
    
        # Make the transcript ID be the symbol _ ensembl ID
        transcript_ID = []
        for idx, sym in enumerate(symbols_unpack):
            ID = sym + "_" + ensembl_unpack[idx]
            transcript_ID.append(ID)
    
        filtered_df = pd.DataFrame(expression_data,index = class_names, columns = transcript_ID)
        filtered_df['cell_identity'] = class_names
    
        # Count occurrences of each cell_identity
        cell_identity_counts = filtered_df['cell_identity'].value_counts()
        print(cell_identity_counts)
    
        # Create a boolean mask to filter rows with cell_identity occurring more than once
        mask = filtered_df['cell_identity'].isin(cell_identity_counts[cell_identity_counts > 1].index)
    
        # Filter the DataFrame using the mask
        filtered_df = filtered_df[mask]
    
        # Seperate the ensmbl ids from the cell_identity column
        TFs = filtered_df.columns.drop(['cell_identity'])
    
        # Set predictors and predicted
        X = np.array(filtered_df[TFs])
        y = np.array(filtered_df['cell_identity'])
    
        # Train - test split with stratification and shuffling
        ### THIS LOADS IN THE WHOLE H5 FILE TO SPLIT
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=5, stratify=y,shuffle = True)
    
        # Create output dataset
        X_train_out = ftrain.create_dataset(
            "expression", 
            (X_train.shape[0], X_train.shape[1]),
            dtype=expression.dtype,
            chunks=True
        )
        print("Starting X Train output into H5")
        chunk_samples(1000,X_train.shape[0],X_train.shape[1],X_train_out,X_train)
        print("X Train output successfully")
    
        y_train_out = ftrain.create_dataset(
            "label",
            len(y_train),
            dtype=h5py.string_dtype(encoding='utf-8'),
            chunks = True
        )
        print("Starting y train output into H5")
        chunk_class(1000,len(y_train),y_train_out,y_train)
        print("y Train output successfully")
    
        TF_ID_out = ftrain.create_dataset(
            "TF_ID",
            len(transcript_ID),
            dtype=h5py.string_dtype(encoding='utf-8'),
            chunks = True
        )
        print("Starting TF IDs output into training H5")
        chunk_class(1000,len(transcript_ID),TF_ID_out,transcript_ID)
        print("TF ID output successfully")
        
        ###export X_test, y_test with Transcript_ID or TFs to an h5
        
        X_test_out = ftest.create_dataset(
            "expression", 
            (X_test.shape[0], X_test.shape[1]),
            dtype=expression.dtype,
            chunks=True
        )
        print("Starting X Test output into H5")
        chunk_samples(1000,X_test.shape[0],X_test.shape[1],X_test_out,X_test)
        print("X test output successfully")
    
        y_test_out = ftest.create_dataset(
            "label",
            len(y_test),
            dtype=h5py.string_dtype(encoding='utf-8'),
            chunks = True
        )
        print("Starting y test output into H5")
        chunk_class(1000,len(y_test),y_test_out,y_test)
        print("y test output successfully")
    
        TF_ID_out = ftest.create_dataset(
            "TF_ID",
            len(transcript_ID),
            dtype=h5py.string_dtype(encoding='utf-8'),
            chunks = True
        )
        print("Starting TF IDs output into testing H5")
        chunk_class(1000,len(transcript_ID),TF_ID_out,transcript_ID)
        print("TF ID output successfully")
    


Expression data has been transposed

cell_identity
Monocyte      5497
Macrophage    4183
Name: count, dtype: int64
Starting X Train output into H5
Iteration 0 (samples 0-1000) is done.
Iteration 1 (samples 1000-2000) is done.
Iteration 2 (samples 2000-3000) is done.
Iteration 3 (samples 3000-4000) is done.
Iteration 4 (samples 4000-5000) is done.
Iteration 5 (samples 5000-6000) is done.
Iteration 6 (samples 6000-7000) is done.
Iteration 7 (samples 7000-7260) is done.
X Train output successfully
Starting y train output into H5
Iteration 0 is done.
Iteration 1 is done.
Iteration 2 is done.
Iteration 3 is done.
Iteration 4 is done.
Iteration 5 is done.
Iteration 6 is done.
Iteration 7 is done.
y Train output successfully
Starting TF IDs output into training H5
Iteration 0 is done.
Iteration 1 is done.
Iteration 2 is done.
Iteration 3 is done.
Iteration 4 is done.
Iteration 5 is done.
Iteration 6 is done.
Iteration 7 is done.
Iteration 8 is done.
Iteration 9 is done.
Iteration 10 is done.


In [17]:
for d in class_sample_dicts:

    mac_key = list(d.keys())[1].replace(" + ","_")
    TF_file_name = "filtered_ARCHS4_train_" + mac_key + ".h5"
    print(TF_file_name)
    a4.ls("/scratch/by2372/pipeline/ARCHS4_Macrophages/Source_name_filtering/" + TF_file_name)
    print("\n")

filtered_ARCHS4_train_Macrophage.h5
TF_ID                 str    | (13146,)
expression            float32 | (7260, 13146)
label                 str    | (7260,)


filtered_ARCHS4_train_Macrophage_Kupffer.h5
TF_ID                 str    | (13146,)
expression            float32 | (7286, 13146)
label                 str    | (7286,)


filtered_ARCHS4_train_Macrophage_Langerhans.h5
TF_ID                 str    | (13146,)
expression            float32 | (7371, 13146)
label                 str    | (7371,)


filtered_ARCHS4_train_Macrophage_Microglia.h5
TF_ID                 str    | (13146,)
expression            float32 | (7764, 13146)
label                 str    | (7764,)


filtered_ARCHS4_train_Macrophage_Osteoclast.h5
TF_ID                 str    | (13146,)
expression            float32 | (7293, 13146)
label                 str    | (7293,)


filtered_ARCHS4_train_Macrophage_alveolar.h5
TF_ID                 str    | (13146,)
expression            float32 | (7406, 13146)
label         

In [18]:
for d in class_sample_dicts:

    mac_key = list(d.keys())[1].replace(" + ","_")
    TF_file_name = "filtered_ARCHS4_test_" + mac_key + ".h5"
    print(TF_file_name)
    a4.ls("/scratch/by2372/pipeline/ARCHS4_Macrophages/Source_name_filtering/" + TF_file_name)
    print("\n")

filtered_ARCHS4_test_Macrophage.h5
TF_ID                 str    | (13146,)
expression            float32 | (2420, 13146)
label                 str    | (2420,)


filtered_ARCHS4_test_Macrophage_Kupffer.h5
TF_ID                 str    | (13146,)
expression            float32 | (2429, 13146)
label                 str    | (2429,)


filtered_ARCHS4_test_Macrophage_Langerhans.h5
TF_ID                 str    | (13146,)
expression            float32 | (2457, 13146)
label                 str    | (2457,)


filtered_ARCHS4_test_Macrophage_Microglia.h5
TF_ID                 str    | (13146,)
expression            float32 | (2588, 13146)
label                 str    | (2588,)


filtered_ARCHS4_test_Macrophage_Osteoclast.h5
TF_ID                 str    | (13146,)
expression            float32 | (2432, 13146)
label                 str    | (2432,)


filtered_ARCHS4_test_Macrophage_alveolar.h5
TF_ID                 str    | (13146,)
expression            float32 | (2469, 13146)
label               

# Below is the code for neural network training

In [10]:
from tensorflow import keras
from sklearn.preprocessing import LabelEncoder
import tensorflow as tef
from keras import optimizers
from keras import losses
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA, KernelPCA
import matplotlib.pyplot as plt
from sklearn.metrics import f1_score
from sklearn.preprocessing import label_binarize
from sklearn.metrics import (auc, roc_curve, roc_auc_score)
from tensorflow.keras.models import load_model
import os

2026-04-28 16:25:55.608057: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-28 16:26:00.205727: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX512_FP16 AVX_VNNI AMX_TILE AMX_INT8 AMX_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-04-28 16:26:10.526783: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [7]:
def NN_training(train_name,test_name,name):
    f_train = h5py.File("/scratch/by2372/pipeline/ARCHS4_Macrophages/Source_name_filtering/" + train_name, 'r')
    X_train = f_train['expression'][...]
    y_train = f_train['label'][...]
    f_train.close()
    f_test = h5py.File("/scratch/by2372/pipeline/ARCHS4_Macrophages/Source_name_filtering/" + test_name, 'r')
    X_test = f_test['expression'][...]
    y_test = f_test['label'][...]
    
    TFs = f_test['TF_ID'][...]
    
    # Read all names and decode bytes to strings
    if isinstance(y_train[0], bytes):
        y_train = [s.decode('utf-8') for s in y_train]
    
    # Read all names and decode bytes to strings
    if isinstance(y_test[0], bytes):
        y_test = [s.decode('utf-8') for s in y_test]
    
    # Read all names and decode bytes to strings
    if isinstance(TFs[0], bytes):
        TFs = [s.decode('utf-8') for s in TFs]
    
    f_test.close()

    keras.utils.set_random_seed(10)   

    le = LabelEncoder()
    
    # Fit on training labels (strings) and transform them to integers
    label_train_int = le.fit_transform(y_train)
    
    # Use the same encoder to transform test labels
    label_test_int = le.transform(y_test)
    
    # Best model in magnussen paper was 2 layers of 250 nodes
    model = keras.Sequential([
        keras.layers.Input(shape=(len(TFs),)),  # there are 13146 TFs
        keras.layers.Dense(250, activation='elu'),
        keras.layers.Dense(250, activation='elu'),
        keras.layers.Dense(1, activation='sigmoid')  # For multiclass classification
    ])

    callback = keras.callbacks.EarlyStopping(monitor='val_loss', patience = 3)
    
    adam = optimizers.Adam(
        learning_rate=0.001,
        beta_1=0.9,
        beta_2=0.999,
        weight_decay=0.01  # replaces old `decay`
    )
    model.compile(loss='binary_crossentropy',optimizer=adam,metrics=['accuracy'], )
    model.fit(X_train, label_train_int, batch_size=50, epochs=100, verbose=1, validation_split = 0.1, callbacks = [callback])

    model.save("/scratch/by2372/pipeline/ARCHS4_Macrophages/Trained_NN/" + name + "_trained_NN.keras")

    np.save("/scratch/by2372/pipeline/ARCHS4_Macrophages/Trained_NN/" + name + "_class_names.npy", le.classes_)

    loss, accuracy = model.evaluate(X_test, label_test_int, verbose=1)

    # Predictions on test set
    y_prob = model.predict(X_test).ravel()
    y_pred = (y_prob >= 0.5).astype(int)

    # Calculate F1 Metrics
    f1 = f1_score(label_test_int, y_pred) 

    fpr, tpr, _ = roc_curve(label_test_int, y_prob)
    roc_auc_val  = auc(fpr, tpr)
    print(f"AUC: {roc_auc_val:.2f}")
    
    return {
    "f1_score": f1,
    "roc_auc": roc_auc_val,
    "loss" : loss,
    "accuracy" : accuracy
    }

In [8]:
for d in class_sample_dicts:

    mac_key = list(d.keys())[1].replace(" + ","_")
    in_train_name = "filtered_ARCHS4_train_" + mac_key + ".h5"
    in_test_name = "filtered_ARCHS4_test_" + mac_key + ".h5"

    metrics = NN_training(in_train_name,in_test_name,mac_key)
    print(mac_key)
    print(metrics)
    print("\n")

2026-04-27 16:31:21.098409: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Epoch 1/100
131/131 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step - accuracy: 0.9126 - loss: 0.8886 - val_accuracy: 0.9587 - val_loss: 0.1461
Epoch 2/100
131/131 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.9656 - loss: 0.1305 - val_accuracy: 0.9780 - val_loss: 0.1107
Epoch 3/100
131/131 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step - accuracy: 0.9835 - loss: 0.0741 - val_accuracy: 0.9780 - val_loss: 0.1109
Epoch 4/100
131/131 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step - accuracy: 0.9868 - loss: 0.0611 - val_accuracy: 0.9793 - val_loss: 0.1053
Epoch 5/100
131/131 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.9847 - loss: 0.0539 - val_accuracy: 0.9835 - val_loss: 0.1091
Epoch 6/100
131/131 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.9861 - loss: 0.0764 - val_accuracy: 0.9807 - val_loss: 0.1389
Epoch 7/100
131/131 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step - accuracy: 0.9815 - loss: 0.0868 - val_accuracy: 0.9477 - val_loss: 0.3430
76/76 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9467 - loss: 0.4319 
76/76 ━━━━━━━━━━

# Node Light up Test is below

In [6]:
# Import expression data and transcript data from filtered h5 file
filtered_file_tf = h5py.File("/scratch/by2372/pipeline/ARCHS4_Macrophages/Source_name_filtering/filtered_ARCHS4_TFs_Macrophage.h5", 'r')
ensembl_id = filtered_file_tf['ensembl_id'][...]
symbol = filtered_file_tf['symbol'][...]
filtered_file_tf.close()

# Read all names and decode bytes to strings
if isinstance(ensembl_id[0], bytes):
    ensembl_id = [s.decode('utf-8') for s in ensembl_id]

# Read all names and decode bytes to strings
if isinstance(symbol[0], bytes):
    symbol = [s.decode('utf-8') for s in symbol]

# Make the transcript ID be the symbol _ ensembl ID
transcript_ID = []
for idx, sym in enumerate(symbol):
    ID = sym + "_" + ensembl_id[idx]
    transcript_ID.append(ID)

# LOOP TO MAKE A DICTIONARY OF FILTERED ARCHS TFS H5 TO BE USED FOR NODE LIGHT UP

expression_dict = dict()

for d in class_sample_dicts:

    mac_key = list(d.keys())[1].replace(" + ","_")
    print(mac_key)
    this_file = h5py.File("/scratch/by2372/pipeline/ARCHS4_Macrophages/Source_name_filtering/filtered_ARCHS4_TFs_" + mac_key + ".h5", 'r')
    expression = this_file['data/filtered_expression'][...]
    this_file.close()
    expression_dict[mac_key] = expression




Macrophage
Macrophage_Kupffer
Macrophage_Langerhans
Macrophage_Microglia
Macrophage_Osteoclast
Macrophage_alveolar


In [7]:
class_names = np.load('/scratch/by2372/pipeline/ARCHS4_Macrophages/Trained_NN/Macrophage_Kupffer_class_names.npy')
class_names

array(['Macrophage + Kupffer', 'Monocyte'], dtype='<U20')

### Implement the Node Light up technique as a function

In [11]:
def node_light_up(model_name,tf_names,express_mat,cell_type):
    # load train keras model
    model = load_model('/scratch/by2372/pipeline/ARCHS4_Macrophages/Trained_NN/' + model_name + '_trained_NN.keras')
    # create square matrix of zeros. Number of rows and cols is number of TFs
    inp = np.zeros((model.input_shape[1],model.input_shape[1])).T
    # Import cell class names 
    class_names = np.load('/scratch/by2372/pipeline/ARCHS4_Macrophages/Trained_NN/' + model_name + '_class_names.npy')

    is_negative_class = (np.where(class_names == cell_type)[0][0] == 0)

    # Generate baseline TF expression values for the inp square matrix
    tf_data = pd.DataFrame(express_mat).iloc[:,:].mean(1)
    for i in range(inp.shape[1]):
        inp[i, :] = tf_data.values
    # Generate doubled and halved expression for TFs. Only diagonals of the baseline square matrix is doubled or halved
    half = inp.copy()
    double = inp.copy()
    half[np.eye(half.shape[0]).astype(bool)] = 0.5*half[np.eye(half.shape[0]).astype(bool)]
    double[np.eye(double.shape[0]).astype(bool)] = 2*double[np.eye(double.shape[0]).astype(bool)]

    background_pred = model.predict(inp)
    double_pred = model.predict(double)
    half_pred = model.predict(half)

    bg_prob     = background_pred[:, 0]
    double_prob = double_pred[:, 0]
    half_prob   = half_pred[:, 0]

    # sigmoid probabilities in binary classification add up to 1
    if is_negative_class:
        bg_prob = 1 - bg_prob
        double_prob = 1 - double_prob
        half_prob = 1 - half_prob

    double_background = double_prob - bg_prob
    half_background   = half_prob   - bg_prob
    
    double_background = pd.DataFrame(double_background, index=tf_names, columns=[cell_type])
    half_background   = pd.DataFrame(half_background,   index=tf_names, columns=[cell_type])

    double_background_sorted = double_background.sort_values(by=cell_type, ascending=True)
    half_background_sorted   = half_background.sort_values(by=cell_type, ascending=False)

    return double_background_sorted, half_background_sorted



# Just Macrophages

In [12]:
macrophage = node_light_up("Macrophage",transcript_ID,expression_dict["Macrophage"],"Macrophage")

2026-04-28 16:26:14.280013: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)
/state/partition1/job-7413004/ipykernel_916059/2910320228.py:12: Pandas4Warning: Starting with pandas version 4.0 all arguments of mean will be keyword-only.
  tf_data = pd.DataFrame(express_mat).iloc[:,:].mean(1)


411/411 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
411/411 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step
411/411 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step 


In [13]:
macrophage[0].head(n=30)

,Macrophage
SPI1_ENST00000378538,-2.384186e-07
CEBPB_ENST00000303004,-2.384186e-07
FOS_ENST00000303562,-2.384186e-07
CSRNP1_ENST00000273153,-1.788139e-07
ATF4_ENST00000674920,-1.788139e-07
KLF6_ENST00000542957,-1.788139e-07
STAT6_ENST00000300134,-1.788139e-07
KMT2A_ENST00000648029,-1.788139e-07
MAFB_ENST00000373313,-1.788139e-07
ZNF302_ENST00000507959,-1.192093e-07


In [14]:
NN = macrophage[0]

for idx, t in enumerate(NN.index):
    if "ZNF385A_ENST00000394313" in str(t):
        print("ZNF385A_ENST00000394313: " + str(idx + 1))

for idx, t in enumerate(NN.index):
    if "CEBPB_ENST00000303004" in str(t):
        print("CEBPB_ENST00000303004: "+ str(idx + 1))

for idx, t in enumerate(NN.index):
    if "CEBPA_ENST00000498907" in str(t):
        print("CEBPA_ENST00000498907: " + str(idx + 1))

for idx, t in enumerate(NN.index):
    if "TSC22D1_ENST00000261489" in str(t):
        print("TSC22D1_ENST00000261489: " + str(idx + 1))

for idx, t in enumerate(NN.index):
    if "ZNF385A_ENST00000546919" in str(t):
        print("ZNF385A_ENST00000546919: " + str(idx + 1))

ZNF385A_ENST00000394313: 13108
CEBPB_ENST00000303004: 2
CEBPA_ENST00000498907: 4010
TSC22D1_ENST00000261489: 13124
ZNF385A_ENST00000546919: 13130


In [15]:
macrophage[1].head(n=30)

,Macrophage
CEBPB_ENST00000303004,2.384186e-07
SPI1_ENST00000378538,1.788139e-07
MAFB_ENST00000373313,1.788139e-07
NFKB1_ENST00000652569,1.192093e-07
CSRNP1_ENST00000273153,1.192093e-07
STAT6_ENST00000300134,1.192093e-07
ATF4_ENST00000674920,1.192093e-07
KMT2A_ENST00000648029,1.192093e-07
KLF6_ENST00000542957,1.192093e-07
FOS_ENST00000556324,1.192093e-07


In [16]:
NN = macrophage[1]

for idx, t in enumerate(NN.index):
    if "ZNF385A_ENST00000394313" in str(t):
        print("ZNF385A_ENST00000394313: " + str(idx + 1))

for idx, t in enumerate(NN.index):
    if "CEBPB_ENST00000303004" in str(t):
        print("CEBPB_ENST00000303004: "+ str(idx + 1))

for idx, t in enumerate(NN.index):
    if "CEBPA_ENST00000498907" in str(t):
        print("CEBPA_ENST00000498907: " + str(idx + 1))

for idx, t in enumerate(NN.index):
    if "TSC22D1_ENST00000261489" in str(t):
        print("TSC22D1_ENST00000261489: " + str(idx + 1))

for idx, t in enumerate(NN.index):
    if "ZNF385A_ENST00000546919" in str(t):
        print("ZNF385A_ENST00000546919: " + str(idx + 1))

ZNF385A_ENST00000394313: 2885
CEBPB_ENST00000303004: 1
CEBPA_ENST00000498907: 3815
TSC22D1_ENST00000261489: 13142
ZNF385A_ENST00000546919: 13121


In [17]:
monocyte = node_light_up("Macrophage",transcript_ID,expression_dict["Macrophage"],"Monocyte")

/state/partition1/job-7413004/ipykernel_916059/2910320228.py:12: Pandas4Warning: Starting with pandas version 4.0 all arguments of mean will be keyword-only.
  tf_data = pd.DataFrame(express_mat).iloc[:,:].mean(1)


411/411 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step  
411/411 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step 
411/411 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step 


In [18]:
monocyte[0].head(n=30)

,Monocyte
NME2_ENST00000512737,-2.026558e-06
YBX1_ENST00000467957,-7.152557e-07
YBX1_ENST00000436427,-5.960464e-07
JUNB_ENST00000302754,-4.172325e-07
JUN_ENST00000678696,-2.980232e-07
GLMP_ENST00000362007,-1.788139e-07
BHLHE41_ENST00000242728,-1.788139e-07
NR1H3_ENST00000441012,-1.788139e-07
MSC_ENST00000325509,-1.788139e-07
ZNF706_ENST00000517844,-1.788139e-07


In [19]:
NN = monocyte[0]

for idx, t in enumerate(NN.index):
    if "ZNF385A_ENST00000394313" in str(t):
        print("ZNF385A_ENST00000394313: " + str(idx + 1))

for idx, t in enumerate(NN.index):
    if "CEBPB_ENST00000303004" in str(t):
        print("CEBPB_ENST00000303004: "+ str(idx + 1))

for idx, t in enumerate(NN.index):
    if "CEBPA_ENST00000498907" in str(t):
        print("CEBPA_ENST00000498907: " + str(idx + 1))

for idx, t in enumerate(NN.index):
    if "TSC22D1_ENST00000261489" in str(t):
        print("TSC22D1_ENST00000261489: " + str(idx + 1))

for idx, t in enumerate(NN.index):
    if "ZNF385A_ENST00000546919" in str(t):
        print("ZNF385A_ENST00000546919: " + str(idx + 1))

ZNF385A_ENST00000394313: 35
CEBPB_ENST00000303004: 13144
CEBPA_ENST00000498907: 3863
TSC22D1_ENST00000261489: 56
ZNF385A_ENST00000546919: 36


In [20]:
monocyte[1].head(n=30)

,Monocyte
NME2_ENST00000512737,1.788139e-07
YBX1_ENST00000467957,1.192093e-07
JUNB_ENST00000302754,1.192093e-07
JUN_ENST00000678696,1.192093e-07
BHLHE40_ENST00000256495,5.960464e-08
ZNF706_ENST00000519744,5.960464e-08
EGR2_ENST00000242480,5.960464e-08
EGR2_ENST00000691610,5.960464e-08
HIF1A_ENST00000555014,5.960464e-08
ETV5_ENST00000306376,5.960464e-08


In [21]:
NN = monocyte[1]

for idx, t in enumerate(NN.index):
    if "ZNF385A_ENST00000394313" in str(t):
        print("ZNF385A_ENST00000394313: " + str(idx + 1))

for idx, t in enumerate(NN.index):
    if "CEBPB_ENST00000303004" in str(t):
        print("CEBPB_ENST00000303004: "+ str(idx + 1))

for idx, t in enumerate(NN.index):
    if "CEBPA_ENST00000498907" in str(t):
        print("CEBPA_ENST00000498907: " + str(idx + 1))

for idx, t in enumerate(NN.index):
    if "TSC22D1_ENST00000261489" in str(t):
        print("TSC22D1_ENST00000261489: " + str(idx + 1))

for idx, t in enumerate(NN.index):
    if "ZNF385A_ENST00000546919" in str(t):
        print("ZNF385A_ENST00000546919: " + str(idx + 1))

ZNF385A_ENST00000394313: 2967
CEBPB_ENST00000303004: 13146
CEBPA_ENST00000498907: 3808
TSC22D1_ENST00000261489: 48
ZNF385A_ENST00000546919: 13


# Macrophage_Kupffer

In [22]:
macrophage_kupffer = node_light_up("Macrophage_Kupffer",transcript_ID,expression_dict["Macrophage_Kupffer"],"Macrophage + Kupffer")

/state/partition1/job-7413004/ipykernel_916059/2910320228.py:12: Pandas4Warning: Starting with pandas version 4.0 all arguments of mean will be keyword-only.
  tf_data = pd.DataFrame(express_mat).iloc[:,:].mean(1)


411/411 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step  
411/411 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step 
411/411 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step


In [23]:
macrophage_kupffer[0].head(n=30)

,Macrophage + Kupffer
JUNB_ENST00000302754,-0.002180
FOS_ENST00000303562,-0.001868
SPI1_ENST00000378538,-0.001732
CEBPB_ENST00000303004,-0.001564
YBX1_ENST00000332220,-0.001551
STAT6_ENST00000300134,-0.001540
KMT2A_ENST00000648029,-0.001212
KLF6_ENST00000497571,-0.001184
KLF6_ENST00000542957,-0.001125
CSRNP1_ENST00000273153,-0.001088


In [24]:
NN = macrophage_kupffer[0]

for idx, t in enumerate(NN.index):
    if "ZNF385A_ENST00000394313" in str(t):
        print("ZNF385A_ENST00000394313: " + str(idx + 1))

for idx, t in enumerate(NN.index):
    if "CEBPB_ENST00000303004" in str(t):
        print("CEBPB_ENST00000303004: "+ str(idx + 1))

for idx, t in enumerate(NN.index):
    if "CEBPA_ENST00000498907" in str(t):
        print("CEBPA_ENST00000498907: " + str(idx + 1))

for idx, t in enumerate(NN.index):
    if "TSC22D1_ENST00000261489" in str(t):
        print("TSC22D1_ENST00000261489: " + str(idx + 1))

for idx, t in enumerate(NN.index):
    if "ZNF385A_ENST00000546919" in str(t):
        print("ZNF385A_ENST00000546919: " + str(idx + 1))

ZNF385A_ENST00000394313: 27
CEBPB_ENST00000303004: 4
CEBPA_ENST00000498907: 12947
TSC22D1_ENST00000261489: 13099
ZNF385A_ENST00000546919: 13096


In [25]:
macrophage_kupffer[1].head(n=30)

,Macrophage + Kupffer
JUNB_ENST00000302754,0.005720
FOS_ENST00000303562,0.003075
SPI1_ENST00000378538,0.002309
CEBPB_ENST00000303004,0.001909
STAT6_ENST00000300134,0.001640
YBX1_ENST00000332220,0.001619
JUND_ENST00000252818,0.001050
KLF6_ENST00000497571,0.001023
KMT2A_ENST00000648029,0.001004
KLF6_ENST00000542957,0.000910


In [26]:
NN = macrophage_kupffer[1]

for idx, t in enumerate(NN.index):
    if "ZNF385A_ENST00000394313" in str(t):
        print("ZNF385A_ENST00000394313: " + str(idx + 1))

for idx, t in enumerate(NN.index):
    if "CEBPB_ENST00000303004" in str(t):
        print("CEBPB_ENST00000303004: "+ str(idx + 1))

for idx, t in enumerate(NN.index):
    if "CEBPA_ENST00000498907" in str(t):
        print("CEBPA_ENST00000498907: " + str(idx + 1))

for idx, t in enumerate(NN.index):
    if "TSC22D1_ENST00000261489" in str(t):
        print("TSC22D1_ENST00000261489: " + str(idx + 1))

for idx, t in enumerate(NN.index):
    if "ZNF385A_ENST00000546919" in str(t):
        print("ZNF385A_ENST00000546919: " + str(idx + 1))

ZNF385A_ENST00000394313: 27
CEBPB_ENST00000303004: 4
CEBPA_ENST00000498907: 12846
TSC22D1_ENST00000261489: 13098
ZNF385A_ENST00000546919: 13097


In [27]:
monocyte_kupffer = node_light_up("Macrophage_Kupffer",transcript_ID,expression_dict["Macrophage_Kupffer"],"Monocyte")

/state/partition1/job-7413004/ipykernel_916059/2910320228.py:12: Pandas4Warning: Starting with pandas version 4.0 all arguments of mean will be keyword-only.
  tf_data = pd.DataFrame(express_mat).iloc[:,:].mean(1)


411/411 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step  
411/411 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step 
411/411 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step 


In [28]:
monocyte_kupffer[0].head(n=30)

,Monocyte
YBX1_ENST00000436427,-0.022159
YBX1_ENST00000467957,-0.011738
NME2_ENST00000512737,-0.008103
JUN_ENST00000678696,-0.004299
MSC_ENST00000325509,-0.002873
SPI1_ENST00000227163,-0.002109
JUN_ENST00000371222,-0.001952
NR1H3_ENST00000441012,-0.001763
IRF1_ENST00000405885,-0.001654
USF2_ENST00000599625,-0.001614


In [29]:
NN = monocyte_kupffer[0]

for idx, t in enumerate(NN.index):
    if "ZNF385A_ENST00000394313" in str(t):
        print("ZNF385A_ENST00000394313: " + str(idx + 1))

for idx, t in enumerate(NN.index):
    if "CEBPB_ENST00000303004" in str(t):
        print("CEBPB_ENST00000303004: "+ str(idx + 1))

for idx, t in enumerate(NN.index):
    if "CEBPA_ENST00000498907" in str(t):
        print("CEBPA_ENST00000498907: " + str(idx + 1))

for idx, t in enumerate(NN.index):
    if "TSC22D1_ENST00000261489" in str(t):
        print("TSC22D1_ENST00000261489: " + str(idx + 1))

for idx, t in enumerate(NN.index):
    if "ZNF385A_ENST00000546919" in str(t):
        print("ZNF385A_ENST00000546919: " + str(idx + 1))

ZNF385A_ENST00000394313: 13120
CEBPB_ENST00000303004: 13143
CEBPA_ENST00000498907: 200
TSC22D1_ENST00000261489: 48
ZNF385A_ENST00000546919: 51


In [30]:
monocyte_kupffer[1].head(n=30)

,Monocyte
YBX1_ENST00000436427,0.001397
YBX1_ENST00000467957,0.001378
JUN_ENST00000678696,0.000932
NME2_ENST00000512737,0.000911
MSC_ENST00000325509,0.000775
SPI1_ENST00000227163,0.000648
JUN_ENST00000371222,0.000600
NR1H3_ENST00000441012,0.000576
USF2_ENST00000599625,0.000544
BHLHE41_ENST00000242728,0.000530


In [31]:
NN = monocyte_kupffer[1]

for idx, t in enumerate(NN.index):
    if "ZNF385A_ENST00000394313" in str(t):
        print("ZNF385A_ENST00000394313: " + str(idx + 1))

for idx, t in enumerate(NN.index):
    if "CEBPB_ENST00000303004" in str(t):
        print("CEBPB_ENST00000303004: "+ str(idx + 1))

for idx, t in enumerate(NN.index):
    if "CEBPA_ENST00000498907" in str(t):
        print("CEBPA_ENST00000498907: " + str(idx + 1))

for idx, t in enumerate(NN.index):
    if "TSC22D1_ENST00000261489" in str(t):
        print("TSC22D1_ENST00000261489: " + str(idx + 1))

for idx, t in enumerate(NN.index):
    if "ZNF385A_ENST00000546919" in str(t):
        print("ZNF385A_ENST00000546919: " + str(idx + 1))

ZNF385A_ENST00000394313: 13120
CEBPB_ENST00000303004: 13143
CEBPA_ENST00000498907: 302
TSC22D1_ENST00000261489: 49
ZNF385A_ENST00000546919: 50


# Macrophage_Langerhans

In [32]:
macrophage_langerhans = node_light_up("Macrophage_Langerhans",transcript_ID,expression_dict["Macrophage_Langerhans"],"Macrophage + Langerhans")

/state/partition1/job-7413004/ipykernel_916059/2910320228.py:12: Pandas4Warning: Starting with pandas version 4.0 all arguments of mean will be keyword-only.
  tf_data = pd.DataFrame(express_mat).iloc[:,:].mean(1)


411/411 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step  
411/411 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step 
411/411 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step 


In [33]:
macrophage_langerhans[0].head(n=30)

,Macrophage + Langerhans
JUNB_ENST00000302754,-0.194267
YBX1_ENST00000332220,-0.162726
CEBPB_ENST00000303004,-0.151314
KMT2A_ENST00000648029,-0.114890
SPI1_ENST00000378538,-0.114455
STAT6_ENST00000300134,-0.109500
JUND_ENST00000252818,-0.102185
FOS_ENST00000556324,-0.088804
ATF4_ENST00000674920,-0.087345
KLF6_ENST00000497571,-0.068182


In [34]:
NN = macrophage_langerhans[0]

for idx, t in enumerate(NN.index):
    if "ZNF385A_ENST00000394313" in str(t):
        print("ZNF385A_ENST00000394313: " + str(idx + 1))

for idx, t in enumerate(NN.index):
    if "CEBPB_ENST00000303004" in str(t):
        print("CEBPB_ENST00000303004: "+ str(idx + 1))

for idx, t in enumerate(NN.index):
    if "CEBPA_ENST00000498907" in str(t):
        print("CEBPA_ENST00000498907: " + str(idx + 1))

for idx, t in enumerate(NN.index):
    if "TSC22D1_ENST00000261489" in str(t):
        print("TSC22D1_ENST00000261489: " + str(idx + 1))

for idx, t in enumerate(NN.index):
    if "ZNF385A_ENST00000546919" in str(t):
        print("ZNF385A_ENST00000546919: " + str(idx + 1))

ZNF385A_ENST00000394313: 528
CEBPB_ENST00000303004: 3
CEBPA_ENST00000498907: 39
TSC22D1_ENST00000261489: 13135
ZNF385A_ENST00000546919: 13056


In [35]:
macrophage_langerhans[1].head(n=30)

,Macrophage + Langerhans
JUNB_ENST00000302754,0.113146
YBX1_ENST00000332220,0.085291
CEBPB_ENST00000303004,0.066283
SPI1_ENST00000378538,0.065700
KMT2A_ENST00000648029,0.062366
STAT6_ENST00000300134,0.059081
JUND_ENST00000252818,0.052022
FOS_ENST00000556324,0.045709
FOS_ENST00000554617,0.035263
FOS_ENST00000535987,0.034648


In [36]:
NN = macrophage_langerhans[1]

for idx, t in enumerate(NN.index):
    if "ZNF385A_ENST00000394313" in str(t):
        print("ZNF385A_ENST00000394313: " + str(idx + 1))

for idx, t in enumerate(NN.index):
    if "CEBPB_ENST00000303004" in str(t):
        print("CEBPB_ENST00000303004: "+ str(idx + 1))

for idx, t in enumerate(NN.index):
    if "CEBPA_ENST00000498907" in str(t):
        print("CEBPA_ENST00000498907: " + str(idx + 1))

for idx, t in enumerate(NN.index):
    if "TSC22D1_ENST00000261489" in str(t):
        print("TSC22D1_ENST00000261489: " + str(idx + 1))

for idx, t in enumerate(NN.index):
    if "ZNF385A_ENST00000546919" in str(t):
        print("ZNF385A_ENST00000546919: " + str(idx + 1))

ZNF385A_ENST00000394313: 237
CEBPB_ENST00000303004: 3
CEBPA_ENST00000498907: 36
TSC22D1_ENST00000261489: 13134
ZNF385A_ENST00000546919: 13055


In [42]:
monocyte_langerhans = node_light_up("Macrophage_Langerhans",transcript_ID,expression_dict["Macrophage_Langerhans"],"Monocyte")

/state/partition1/job-7413004/ipykernel_916059/2910320228.py:12: Pandas4Warning: Starting with pandas version 4.0 all arguments of mean will be keyword-only.
  tf_data = pd.DataFrame(express_mat).iloc[:,:].mean(1)


411/411 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step  
411/411 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step 
411/411 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step 


In [43]:
monocyte_langerhans[0].head(n=30)

,Monocyte
YBX1_ENST00000436427,-0.359798
ATF3_ENST00000341491,-0.200466
MAFB_ENST00000373313,-0.155955
JUN_ENST00000678696,-0.152309
YBX1_ENST00000467957,-0.132948
FOS_ENST00000303562,-0.112560
JUN_ENST00000371222,-0.106706
MSC_ENST00000325509,-0.092874
EGR1_ENST00000239938,-0.073751
GLMP_ENST00000362007,-0.073604


In [44]:
NN = monocyte_langerhans[0]

for idx, t in enumerate(NN.index):
    if "ZNF385A_ENST00000394313" in str(t):
        print("ZNF385A_ENST00000394313: " + str(idx + 1))

for idx, t in enumerate(NN.index):
    if "CEBPB_ENST00000303004" in str(t):
        print("CEBPB_ENST00000303004: "+ str(idx + 1))

for idx, t in enumerate(NN.index):
    if "CEBPA_ENST00000498907" in str(t):
        print("CEBPA_ENST00000498907: " + str(idx + 1))

for idx, t in enumerate(NN.index):
    if "TSC22D1_ENST00000261489" in str(t):
        print("TSC22D1_ENST00000261489: " + str(idx + 1))

for idx, t in enumerate(NN.index):
    if "ZNF385A_ENST00000546919" in str(t):
        print("ZNF385A_ENST00000546919: " + str(idx + 1))

ZNF385A_ENST00000394313: 12619
CEBPB_ENST00000303004: 13144
CEBPA_ENST00000498907: 13108
TSC22D1_ENST00000261489: 12
ZNF385A_ENST00000546919: 91


In [45]:
monocyte_langerhans[1].head(n=30)

,Monocyte
YBX1_ENST00000436427,0.173720
ATF3_ENST00000341491,0.102863
MAFB_ENST00000373313,0.077917
JUN_ENST00000678696,0.069995
YBX1_ENST00000467957,0.058241
JUN_ENST00000371222,0.053094
FOS_ENST00000303562,0.052882
MSC_ENST00000325509,0.045450
GLMP_ENST00000362007,0.035901
EGR1_ENST00000239938,0.035748


In [46]:
NN = monocyte_langerhans[1]

for idx, t in enumerate(NN.index):
    if "ZNF385A_ENST00000394313" in str(t):
        print("ZNF385A_ENST00000394313: " + str(idx + 1))

for idx, t in enumerate(NN.index):
    if "CEBPB_ENST00000303004" in str(t):
        print("CEBPB_ENST00000303004: "+ str(idx + 1))

for idx, t in enumerate(NN.index):
    if "CEBPA_ENST00000498907" in str(t):
        print("CEBPA_ENST00000498907: " + str(idx + 1))

for idx, t in enumerate(NN.index):
    if "TSC22D1_ENST00000261489" in str(t):
        print("TSC22D1_ENST00000261489: " + str(idx + 1))

for idx, t in enumerate(NN.index):
    if "ZNF385A_ENST00000546919" in str(t):
        print("ZNF385A_ENST00000546919: " + str(idx + 1))

ZNF385A_ENST00000394313: 12910
CEBPB_ENST00000303004: 13144
CEBPA_ENST00000498907: 13111
TSC22D1_ENST00000261489: 13
ZNF385A_ENST00000546919: 92


# Macrophage_Microglia

In [37]:
macrophage_microglia = node_light_up("Macrophage_Microglia",transcript_ID,expression_dict["Macrophage_Microglia"],"Macrophage + Microglia")

/state/partition1/job-7413004/ipykernel_916059/2910320228.py:12: Pandas4Warning: Starting with pandas version 4.0 all arguments of mean will be keyword-only.
  tf_data = pd.DataFrame(express_mat).iloc[:,:].mean(1)


411/411 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step  
411/411 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step 
411/411 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step 


In [38]:
macrophage_microglia[0].head(n=30)

,Macrophage + Microglia
JUNB_ENST00000302754,-0.226869
SPI1_ENST00000378538,-0.167244
YBX1_ENST00000332220,-0.165452
KLF6_ENST00000542957,-0.121248
STAT6_ENST00000300134,-0.105704
YBX1_ENST00000321358,-0.095035
MAFB_ENST00000373313,-0.087585
USF2_ENST00000222305,-0.084501
KLF6_ENST00000497571,-0.081696
DRAP1_ENST00000312515,-0.074515


In [39]:
NN = macrophage_microglia[0]

for idx, t in enumerate(NN.index):
    if "ZNF385A_ENST00000394313" in str(t):
        print("ZNF385A_ENST00000394313: " + str(idx + 1))

for idx, t in enumerate(NN.index):
    if "CEBPB_ENST00000303004" in str(t):
        print("CEBPB_ENST00000303004: "+ str(idx + 1))

for idx, t in enumerate(NN.index):
    if "CEBPA_ENST00000498907" in str(t):
        print("CEBPA_ENST00000498907: " + str(idx + 1))

for idx, t in enumerate(NN.index):
    if "TSC22D1_ENST00000261489" in str(t):
        print("TSC22D1_ENST00000261489: " + str(idx + 1))

for idx, t in enumerate(NN.index):
    if "ZNF385A_ENST00000546919" in str(t):
        print("ZNF385A_ENST00000546919: " + str(idx + 1))

ZNF385A_ENST00000394313: 12
CEBPB_ENST00000303004: 12966
CEBPA_ENST00000498907: 46
TSC22D1_ENST00000261489: 13010
ZNF385A_ENST00000546919: 13098


In [40]:
macrophage_microglia[1].head(n=30)

,Macrophage + Microglia
JUNB_ENST00000302754,0.147469
YBX1_ENST00000332220,0.111897
SPI1_ENST00000378538,0.104239
KLF6_ENST00000542957,0.076457
STAT6_ENST00000300134,0.061010
YBX1_ENST00000321358,0.054091
USF2_ENST00000222305,0.048201
ATF4_ENST00000674920,0.047551
KLF6_ENST00000497571,0.046008
DRAP1_ENST00000312515,0.041744


In [41]:
NN = macrophage_microglia[1]

for idx, t in enumerate(NN.index):
    if "ZNF385A_ENST00000394313" in str(t):
        print("ZNF385A_ENST00000394313: " + str(idx + 1))

for idx, t in enumerate(NN.index):
    if "CEBPB_ENST00000303004" in str(t):
        print("CEBPB_ENST00000303004: "+ str(idx + 1))

for idx, t in enumerate(NN.index):
    if "CEBPA_ENST00000498907" in str(t):
        print("CEBPA_ENST00000498907: " + str(idx + 1))

for idx, t in enumerate(NN.index):
    if "TSC22D1_ENST00000261489" in str(t):
        print("TSC22D1_ENST00000261489: " + str(idx + 1))

for idx, t in enumerate(NN.index):
    if "ZNF385A_ENST00000546919" in str(t):
        print("ZNF385A_ENST00000546919: " + str(idx + 1))

ZNF385A_ENST00000394313: 13
CEBPB_ENST00000303004: 13065
CEBPA_ENST00000498907: 44
TSC22D1_ENST00000261489: 13032
ZNF385A_ENST00000546919: 13096


In [47]:
monocyte_microglia = node_light_up("Macrophage_Microglia",transcript_ID,expression_dict["Macrophage_Microglia"],"Monocyte")

/state/partition1/job-7413004/ipykernel_916059/2910320228.py:12: Pandas4Warning: Starting with pandas version 4.0 all arguments of mean will be keyword-only.
  tf_data = pd.DataFrame(express_mat).iloc[:,:].mean(1)


411/411 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step  
411/411 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step 
411/411 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step 


In [48]:
monocyte_microglia[0].head(n=30)

,Monocyte
YBX1_ENST00000467957,-0.256633
YBX1_ENST00000436427,-0.252237
NME2_ENST00000512737,-0.210939
NFE2L2_ENST00000397062,-0.164664
MSC_ENST00000325509,-0.124474
ZNF706_ENST00000519744,-0.102263
BHLHE41_ENST00000242728,-0.090667
ETV5_ENST00000306376,-0.085146
GLMP_ENST00000362007,-0.084551
JUN_ENST00000678696,-0.080724


In [49]:
NN = monocyte_microglia[0]

for idx, t in enumerate(NN.index):
    if "ZNF385A_ENST00000394313" in str(t):
        print("ZNF385A_ENST00000394313: " + str(idx + 1))

for idx, t in enumerate(NN.index):
    if "CEBPB_ENST00000303004" in str(t):
        print("CEBPB_ENST00000303004: "+ str(idx + 1))

for idx, t in enumerate(NN.index):
    if "CEBPA_ENST00000498907" in str(t):
        print("CEBPA_ENST00000498907: " + str(idx + 1))

for idx, t in enumerate(NN.index):
    if "TSC22D1_ENST00000261489" in str(t):
        print("TSC22D1_ENST00000261489: " + str(idx + 1))

for idx, t in enumerate(NN.index):
    if "ZNF385A_ENST00000546919" in str(t):
        print("ZNF385A_ENST00000546919: " + str(idx + 1))

ZNF385A_ENST00000394313: 13135
CEBPB_ENST00000303004: 181
CEBPA_ENST00000498907: 13101
TSC22D1_ENST00000261489: 137
ZNF385A_ENST00000546919: 49


In [50]:
monocyte_microglia[1].head(n=30)

,Monocyte
YBX1_ENST00000467957,0.103226
YBX1_ENST00000436427,0.095759
NME2_ENST00000512737,0.080932
NFE2L2_ENST00000397062,0.068379
MSC_ENST00000325509,0.055755
ZNF706_ENST00000519744,0.046773
BHLHE41_ENST00000242728,0.041176
XBP1_ENST00000216037,0.038593
GLMP_ENST00000362007,0.038136
ETV5_ENST00000306376,0.038118


In [51]:
NN = monocyte_microglia[1]

for idx, t in enumerate(NN.index):
    if "ZNF385A_ENST00000394313" in str(t):
        print("ZNF385A_ENST00000394313: " + str(idx + 1))

for idx, t in enumerate(NN.index):
    if "CEBPB_ENST00000303004" in str(t):
        print("CEBPB_ENST00000303004: "+ str(idx + 1))

for idx, t in enumerate(NN.index):
    if "CEBPA_ENST00000498907" in str(t):
        print("CEBPA_ENST00000498907: " + str(idx + 1))

for idx, t in enumerate(NN.index):
    if "TSC22D1_ENST00000261489" in str(t):
        print("TSC22D1_ENST00000261489: " + str(idx + 1))

for idx, t in enumerate(NN.index):
    if "ZNF385A_ENST00000546919" in str(t):
        print("ZNF385A_ENST00000546919: " + str(idx + 1))

ZNF385A_ENST00000394313: 13134
CEBPB_ENST00000303004: 82
CEBPA_ENST00000498907: 13103
TSC22D1_ENST00000261489: 115
ZNF385A_ENST00000546919: 51


# Macrophage_Osteoclast

In [56]:
macrophage_osteoclast = node_light_up("Macrophage_Osteoclast",transcript_ID,expression_dict["Macrophage_Osteoclast"],"Macrophage + Osteoclast")

/state/partition1/job-7404560/ipykernel_2377561/2910320228.py:12: Pandas4Warning: Starting with pandas version 4.0 all arguments of mean will be keyword-only.
  tf_data = pd.DataFrame(express_mat).iloc[:,:].mean(1)


411/411 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step  
411/411 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step 
411/411 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step 


In [57]:
macrophage_osteoclast[0].head(n=30)

,Macrophage + Osteoclast
FOS_ENST00000303562,-0.000433
JUNB_ENST00000302754,-0.000390
SPI1_ENST00000378538,-0.000387
YBX1_ENST00000332220,-0.000361
CEBPB_ENST00000303004,-0.000275
ATF4_ENST00000674920,-0.000264
KLF6_ENST00000542957,-0.000262
STAT6_ENST00000300134,-0.000260
FOS_ENST00000554617,-0.000247
FOS_ENST00000556324,-0.000238


In [58]:
NN = macrophage_osteoclast[0]

for idx, t in enumerate(NN.index):
    if "ZNF385A_ENST00000394313" in str(t):
        print("ZNF385A_ENST00000394313: " + str(idx + 1))

for idx, t in enumerate(NN.index):
    if "CEBPB_ENST00000303004" in str(t):
        print("CEBPB_ENST00000303004: "+ str(idx + 1))

for idx, t in enumerate(NN.index):
    if "CEBPA_ENST00000498907" in str(t):
        print("CEBPA_ENST00000498907: " + str(idx + 1))

for idx, t in enumerate(NN.index):
    if "TSC22D1_ENST00000261489" in str(t):
        print("TSC22D1_ENST00000261489: " + str(idx + 1))

for idx, t in enumerate(NN.index):
    if "ZNF385A_ENST00000546919" in str(t):
        print("ZNF385A_ENST00000546919: " + str(idx + 1))

ZNF385A_ENST00000394313: 185
CEBPB_ENST00000303004: 5
CEBPA_ENST00000498907: 62
TSC22D1_ENST00000261489: 13086
ZNF385A_ENST00000546919: 13101


In [59]:
macrophage_osteoclast[1].head(n=30)

,Macrophage + Osteoclast
FOS_ENST00000303562,0.000469
SPI1_ENST00000378538,0.000403
YBX1_ENST00000332220,0.000395
JUNB_ENST00000302754,0.000330
CEBPB_ENST00000303004,0.000220
STAT6_ENST00000300134,0.000211
KLF6_ENST00000542957,0.000210
ATF4_ENST00000674920,0.000202
FOS_ENST00000554617,0.000190
FOS_ENST00000556324,0.000180


In [60]:
NN = macrophage_osteoclast[1]

for idx, t in enumerate(NN.index):
    if "ZNF385A_ENST00000394313" in str(t):
        print("ZNF385A_ENST00000394313: " + str(idx + 1))

for idx, t in enumerate(NN.index):
    if "CEBPB_ENST00000303004" in str(t):
        print("CEBPB_ENST00000303004: "+ str(idx + 1))

for idx, t in enumerate(NN.index):
    if "CEBPA_ENST00000498907" in str(t):
        print("CEBPA_ENST00000498907: " + str(idx + 1))

for idx, t in enumerate(NN.index):
    if "TSC22D1_ENST00000261489" in str(t):
        print("TSC22D1_ENST00000261489: " + str(idx + 1))

for idx, t in enumerate(NN.index):
    if "ZNF385A_ENST00000546919" in str(t):
        print("ZNF385A_ENST00000546919: " + str(idx + 1))

ZNF385A_ENST00000394313: 269
CEBPB_ENST00000303004: 5
CEBPA_ENST00000498907: 61
TSC22D1_ENST00000261489: 13088
ZNF385A_ENST00000546919: 13101


In [52]:
monocyte_osteoclast = node_light_up("Macrophage_Osteoclast",transcript_ID,expression_dict["Macrophage_Osteoclast"],"Monocyte")

/state/partition1/job-7413004/ipykernel_916059/2910320228.py:12: Pandas4Warning: Starting with pandas version 4.0 all arguments of mean will be keyword-only.
  tf_data = pd.DataFrame(express_mat).iloc[:,:].mean(1)


411/411 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step  
411/411 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step 
411/411 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step 


In [53]:
monocyte_osteoclast[0].head(n=30)

,Monocyte
YBX1_ENST00000436427,-0.002672
JUN_ENST00000678696,-0.000739
YBX1_ENST00000467957,-0.000697
SPI1_ENST00000227163,-0.000631
NME2_ENST00000512737,-0.000576
NR1H3_ENST00000441012,-0.000320
ZNF706_ENST00000517844,-0.000317
ZNF706_ENST00000519744,-0.000310
ATF3_ENST00000341491,-0.000293
NFE2L2_ENST00000397062,-0.000277


In [54]:
NN = monocyte_osteoclast[0]

for idx, t in enumerate(NN.index):
    if "ZNF385A_ENST00000394313" in str(t):
        print("ZNF385A_ENST00000394313: " + str(idx + 1))

for idx, t in enumerate(NN.index):
    if "CEBPB_ENST00000303004" in str(t):
        print("CEBPB_ENST00000303004: "+ str(idx + 1))

for idx, t in enumerate(NN.index):
    if "CEBPA_ENST00000498907" in str(t):
        print("CEBPA_ENST00000498907: " + str(idx + 1))

for idx, t in enumerate(NN.index):
    if "TSC22D1_ENST00000261489" in str(t):
        print("TSC22D1_ENST00000261489: " + str(idx + 1))

for idx, t in enumerate(NN.index):
    if "ZNF385A_ENST00000546919" in str(t):
        print("ZNF385A_ENST00000546919: " + str(idx + 1))

ZNF385A_ENST00000394313: 12963
CEBPB_ENST00000303004: 13142
CEBPA_ENST00000498907: 13085
TSC22D1_ENST00000261489: 61
ZNF385A_ENST00000546919: 46


In [55]:
monocyte_osteoclast[1].head(n=30)

,Monocyte
YBX1_ENST00000436427,0.000357
JUN_ENST00000678696,0.000189
YBX1_ENST00000467957,0.000183
NME2_ENST00000512737,0.000181
SPI1_ENST00000227163,0.000170
ZNF706_ENST00000517844,0.000114
NR1H3_ENST00000441012,0.000112
ZNF706_ENST00000519744,0.000110
ATF3_ENST00000341491,0.000105
BHLHE41_ENST00000242728,0.000100


In [56]:
NN = monocyte_osteoclast[1]

for idx, t in enumerate(NN.index):
    if "ZNF385A_ENST00000394313" in str(t):
        print("ZNF385A_ENST00000394313: " + str(idx + 1))

for idx, t in enumerate(NN.index):
    if "CEBPB_ENST00000303004" in str(t):
        print("CEBPB_ENST00000303004: "+ str(idx + 1))

for idx, t in enumerate(NN.index):
    if "CEBPA_ENST00000498907" in str(t):
        print("CEBPA_ENST00000498907: " + str(idx + 1))

for idx, t in enumerate(NN.index):
    if "TSC22D1_ENST00000261489" in str(t):
        print("TSC22D1_ENST00000261489: " + str(idx + 1))

for idx, t in enumerate(NN.index):
    if "ZNF385A_ENST00000546919" in str(t):
        print("ZNF385A_ENST00000546919: " + str(idx + 1))

ZNF385A_ENST00000394313: 12879
CEBPB_ENST00000303004: 13142
CEBPA_ENST00000498907: 13086
TSC22D1_ENST00000261489: 59
ZNF385A_ENST00000546919: 46


# Macrophage_alveolar

In [61]:
macrophage_alveolar = node_light_up("Macrophage_alveolar",transcript_ID,expression_dict["Macrophage_alveolar"],"Macrophage + alveolar")

/state/partition1/job-7404560/ipykernel_2377561/2910320228.py:12: Pandas4Warning: Starting with pandas version 4.0 all arguments of mean will be keyword-only.
  tf_data = pd.DataFrame(express_mat).iloc[:,:].mean(1)


411/411 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step  
411/411 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step 
411/411 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step 


In [62]:
macrophage_alveolar[0].head(n=30)

,Macrophage + alveolar
CEBPB_ENST00000303004,-0.007888
FOS_ENST00000556324,-0.006688
ATF4_ENST00000674920,-0.006655
DRAP1_ENST00000312515,-0.005198
KMT2A_ENST00000648029,-0.005064
JUNB_ENST00000302754,-0.004462
YBX1_ENST00000332220,-0.004282
SPI1_ENST00000378538,-0.004195
FOSL2_ENST00000264716,-0.004099
USF2_ENST00000222305,-0.004065


In [63]:
NN = macrophage_alveolar[0]

for idx, t in enumerate(NN.index):
    if "ZNF385A_ENST00000394313" in str(t):
        print("ZNF385A_ENST00000394313: " + str(idx + 1))

for idx, t in enumerate(NN.index):
    if "CEBPB_ENST00000303004" in str(t):
        print("CEBPB_ENST00000303004: "+ str(idx + 1))

for idx, t in enumerate(NN.index):
    if "CEBPA_ENST00000498907" in str(t):
        print("CEBPA_ENST00000498907: " + str(idx + 1))

for idx, t in enumerate(NN.index):
    if "TSC22D1_ENST00000261489" in str(t):
        print("TSC22D1_ENST00000261489: " + str(idx + 1))

for idx, t in enumerate(NN.index):
    if "ZNF385A_ENST00000546919" in str(t):
        print("ZNF385A_ENST00000546919: " + str(idx + 1))

ZNF385A_ENST00000394313: 17
CEBPB_ENST00000303004: 1
CEBPA_ENST00000498907: 20
TSC22D1_ENST00000261489: 12809
ZNF385A_ENST00000546919: 13100


In [64]:
macrophage_alveolar[1].head(n=30)

,Macrophage + alveolar
ATF4_ENST00000674920,0.004592
FOS_ENST00000556324,0.004195
DRAP1_ENST00000312515,0.003064
CEBPB_ENST00000303004,0.003046
KMT2A_ENST00000648029,0.002670
NFKB1_ENST00000652569,0.002329
USF2_ENST00000222305,0.002237
STAT6_ENST00000300134,0.002205
FOSL2_ENST00000264716,0.002129
HMG20B_ENST00000333651,0.002073


In [65]:
NN = macrophage_alveolar[1]

for idx, t in enumerate(NN.index):
    if "ZNF385A_ENST00000394313" in str(t):
        print("ZNF385A_ENST00000394313: " + str(idx + 1))

for idx, t in enumerate(NN.index):
    if "CEBPB_ENST00000303004" in str(t):
        print("CEBPB_ENST00000303004: "+ str(idx + 1))

for idx, t in enumerate(NN.index):
    if "CEBPA_ENST00000498907" in str(t):
        print("CEBPA_ENST00000498907: " + str(idx + 1))

for idx, t in enumerate(NN.index):
    if "TSC22D1_ENST00000261489" in str(t):
        print("TSC22D1_ENST00000261489: " + str(idx + 1))

for idx, t in enumerate(NN.index):
    if "ZNF385A_ENST00000546919" in str(t):
        print("ZNF385A_ENST00000546919: " + str(idx + 1))

ZNF385A_ENST00000394313: 13
CEBPB_ENST00000303004: 4
CEBPA_ENST00000498907: 15
TSC22D1_ENST00000261489: 12984
ZNF385A_ENST00000546919: 13096


In [57]:
monocyte_alveolar = node_light_up("Macrophage_alveolar",transcript_ID,expression_dict["Macrophage_alveolar"],"Monocyte")

/state/partition1/job-7413004/ipykernel_916059/2910320228.py:12: Pandas4Warning: Starting with pandas version 4.0 all arguments of mean will be keyword-only.
  tf_data = pd.DataFrame(express_mat).iloc[:,:].mean(1)


411/411 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step  
411/411 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step 
411/411 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step 


In [58]:
monocyte_alveolar[0].head(n=30)

,Monocyte
YBX1_ENST00000436427,-0.050885
NME2_ENST00000512737,-0.022949
YBX1_ENST00000467957,-0.014755
SPI1_ENST00000227163,-0.008060
MSC_ENST00000325509,-0.006215
NFE2L2_ENST00000397062,-0.005181
GLMP_ENST00000362007,-0.004190
ZNF706_ENST00000519744,-0.004095
BHLHE41_ENST00000242728,-0.003959
NR1H3_ENST00000441012,-0.003340


In [59]:
NN = monocyte_alveolar[0]

for idx, t in enumerate(NN.index):
    if "ZNF385A_ENST00000394313" in str(t):
        print("ZNF385A_ENST00000394313: " + str(idx + 1))

for idx, t in enumerate(NN.index):
    if "CEBPB_ENST00000303004" in str(t):
        print("CEBPB_ENST00000303004: "+ str(idx + 1))

for idx, t in enumerate(NN.index):
    if "CEBPA_ENST00000498907" in str(t):
        print("CEBPA_ENST00000498907: " + str(idx + 1))

for idx, t in enumerate(NN.index):
    if "TSC22D1_ENST00000261489" in str(t):
        print("TSC22D1_ENST00000261489: " + str(idx + 1))

for idx, t in enumerate(NN.index):
    if "ZNF385A_ENST00000546919" in str(t):
        print("ZNF385A_ENST00000546919: " + str(idx + 1))

ZNF385A_ENST00000394313: 13130
CEBPB_ENST00000303004: 13146
CEBPA_ENST00000498907: 13127
TSC22D1_ENST00000261489: 338
ZNF385A_ENST00000546919: 47


In [60]:
monocyte_alveolar[1].head(n=30)

,Monocyte
YBX1_ENST00000436427,0.009485
NME2_ENST00000512737,0.008006
YBX1_ENST00000467957,0.004974
JUNB_ENST00000302754,0.003595
SPI1_ENST00000227163,0.003352
MSC_ENST00000325509,0.002814
NFE2L2_ENST00000397062,0.002257
ATF3_ENST00000341491,0.001962
GLMP_ENST00000362007,0.001879
BHLHE41_ENST00000242728,0.001868


In [61]:
NN = monocyte_alveolar[1]

for idx, t in enumerate(NN.index):
    if "ZNF385A_ENST00000394313" in str(t):
        print("ZNF385A_ENST00000394313: " + str(idx + 1))

for idx, t in enumerate(NN.index):
    if "CEBPB_ENST00000303004" in str(t):
        print("CEBPB_ENST00000303004: "+ str(idx + 1))

for idx, t in enumerate(NN.index):
    if "CEBPA_ENST00000498907" in str(t):
        print("CEBPA_ENST00000498907: " + str(idx + 1))

for idx, t in enumerate(NN.index):
    if "TSC22D1_ENST00000261489" in str(t):
        print("TSC22D1_ENST00000261489: " + str(idx + 1))

for idx, t in enumerate(NN.index):
    if "ZNF385A_ENST00000546919" in str(t):
        print("ZNF385A_ENST00000546919: " + str(idx + 1))

ZNF385A_ENST00000394313: 13134
CEBPB_ENST00000303004: 13143
CEBPA_ENST00000498907: 13132
TSC22D1_ENST00000261489: 163
ZNF385A_ENST00000546919: 51
